# 00 · 接口探索

## 这个 notebook 在做什么？

你以前爬 KPL 数据，一上来就开始写循环、写保存。这次咱们换个做法——**第一步不爬数据，而是「摸数据」**。

为什么？你上一个项目里有个老问题：写了一堆字段（`game_player_camp1.csv` 那些），但很多字段你自己都说不清是什么意思、单位是什么、有没有缺失。这是因为**没有先做接口探索**就开干了。

企业里的数据项目，第一周往往不写一行业务代码，只干一件事：**把字段地图画清楚**。

## 这个 notebook 跑完，你应该收获什么？

1. 一份原始 JSON 样本（存在 `../data/raw/sample.json`）——后面所有探索都基于它，不用反复请求接口
2. 对接口 JSON 的「四级结构」心里有数：**比赛级 → 战队级 → 选手级 → BP 级**
3. 验证 `battle_id` 的编码规则——你之前的猜测对不对？
4. 找到赛程接口（这是后面批量爬的入口）
5. 一份填好的 `docs/数据字典.md`

## 建议怎么做

- 一个 cell 一个 cell 顺着往下做
- 每一步先看「思路」再写代码，**别看到 TODO 就直接想答案**
- 卡住先记下来继续往下走，最后一起问我
- 每个步骤后面都有「💡 你的观察」markdown，**这部分一定要写**——这就是你简历上「我做了什么思考」的素材

---
## 步骤 1 · 准备工作

### 思路

做接口探索需要两类东西：
1. **工具包**：`requests` 调接口、`json` 处理 JSON、`pandas` 看表（DataFrame 比 dict 好读 100 倍）、`pathlib` 处理路径
2. **常量**：URL、Headers、一个测试用的 battle_id

为什么把常量提出来？因为后面会反复用——如果写在每个 cell 里，改一次要改 N 处。

> 这个习惯将来要一直保持：**只要某个值会出现 2 次以上，就提出来当常量**。

In [ ]:
# TODO 1.1：导入 requests / json / pandas / pathlib.Path
# 提示：pandas 习惯写 import pandas as pd
import requests
import json
import pandas as pd
from pathlib import Path

from pandas import json_normalize


In [ ]:
# TODO 1.2：定义三个常量
# - BASE_URL：单场比赛接口地址（不带 ?battle_id=xxx 那部分）
#   提示：去看 scripts/01api.py，那里已经验证过的就是这个
# - HEADERS：必需的请求头
#   思考：为什么需要 Referer / Origin？（提示：腾讯做了反爬，没这俩头会 403）
# - TEST_BATTLE_ID：拿一个你已经验证过能用的 battle_id

BASE_URL = "https://prod.comp.smoba.qq.com/leaguesite/battle/open"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/128.0.0.0 Mobile Safari/537.36 Edg/128.0.0.0",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://pvp.qq.com/",
    "Origin": "https://pvp.qq.com",
}
TEST_BATTLE_ID = "736117264_10_1777089898"


---
## 步骤 2 · 调一次接口，把原始 JSON 存到本地

### 思路

**永远先把原始数据存一份在本地**——这是企业级项目的铁律，理由有三：

1. **防被封**：探索阶段会反复看字段，每看一次就请求一次接口，几百次以后就被反爬封 IP 了
2. **可复现**：你存的那一份永远不变，几个月后回来研究依然能跑
3. **可对比**：万一接口字段变了，能跟旧版本 diff

存的格式：JSON 原样存，用 `ensure_ascii=False` 让中文不变成 `\u5f20\u4e09` 这种乱码。

In [ ]:
# TODO 2.1：用 requests.get() 请求接口
# 思考：
#   - URL 怎么拼？f-string 一下：f"{BASE_URL}?battle_id={TEST_BATTLE_ID}"
#   - timeout 设多少？（建议 15 秒，太短会误报失败，太长卡顿）
#   - 怎么把响应转成字典？.json() 方法
# 把结果存到变量 data 里

url = f"{BASE_URL}?battle_id={TEST_BATTLE_ID}"
resp =  requests.get(url,headers=HEADERS,timeout=15)
data = resp.json()

# 自检：data 应该是个 dict，code 应该是 200
print(type(data), data.get('code'))

url1 = "https://prod.comp.smoba.qq.com/leaguesite/battle/open?battle_id=736117264_12_1777093747"
resp = requests.get(url1,headers=HEADERS,timeout=15)
data1 = resp.json()

<class 'dict'> 200


In [ ]:
# TODO 2.2：把 data 存到 ../data/raw/sample.json
# 提示：
#   - 先用 Path('../data/raw') 拿到目录
#   - .mkdir(parents=True, exist_ok=True) 保证目录存在
#   - 写文件用 with open(path, 'w', encoding='utf-8') as f:
#   - json.dump(data, f, ensure_ascii=False, indent=2)
#         ↑ 这俩参数缺一不可：ensure_ascii=False 中文不乱码，indent=2 文件可读
# 写完打印保存路径 + 文件大小（KB 级别），证明写入成功

project_root = Path('../data/raw').resolve()
print(f'project_root:{project_root}')
path = project_root/"sample.json"
if not project_root.exists():
	project_root.mkdir(parents=True, exist_ok=True)
else:
	pass
with open(path, "w",encoding="utf-8-sig") as f:
	json.dump(data,f,ensure_ascii=False,indent=2)

print(f"保存路径：{project_root}")

project_root:D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\raw
保存路径：D:\AI数据分析\项目作品集\02-KPL实时胜率预测系统\data\raw


---
## 步骤 3 · 剥洋葱第一层：看顶层结构

### 思路

JSON 是**嵌套结构**——一层套一层。探索的方法叫「剥洋葱」：

```
data                       ← 最外层
├── code, message          ← 接口元信息
└── data                   ← 真正的业务数据（注意命名重复了）
    ├── battle_id, status  ← 比赛级字段
    ├── camp1, camp2       ← 战队级字段（dict）
    ├── battle_player_list ← 选手级字段（list of 10 个 dict）
    └── bp_list            ← BP 级字段（list of 多个 dict）
```

这一步只看**最外层**有哪些 key。

In [ ]:
# TODO 3.1：打印 data 的顶层 keys
# 提示：dict.keys() 或 list(dict)
print(data.keys())

dict_keys(['message', 'code', 'request_id', 'data'])


In [ ]:
data['data'].keys()

dict_keys(['battle_id', 'status', 'win_camp', 'game_duration', 'battle_seq', 'camp1', 'camp2', 'battle_player_list', 'bp_list', 'video_list'])

In [ ]:
# TODO 3.2：打印 data['data'] 的二级 keys
# 然后给自己一个直观的认识：
#   - 大致有多少个字段？
#   - 哪些字段名你一看就懂？哪些猜不出？（猜不出的等会儿挨个看）
data['data']['camp1'].keys()

dict_keys(['team_id', 'team_name', 'team_icon', 'is_win', 'kill_num', 'assist_num', 'death_num', 'gold', 'kill_big_dragon_num', 'push_tower_num', 'kill_dark_tyrant_num', 'kill_tyrant_num', 'kda', 'kill_prophet_dragon_num', 'kill_shadow_dragon_num', 'kill_storm_dragon_king_num', 'team_abbreviation'])

In [ ]:
data['data']['camp2'].keys()

dict_keys(['team_id', 'team_name', 'team_icon', 'is_win', 'kill_num', 'assist_num', 'death_num', 'gold', 'kill_big_dragon_num', 'push_tower_num', 'kill_dark_tyrant_num', 'kill_tyrant_num', 'kda', 'kill_prophet_dragon_num', 'kill_shadow_dragon_num', 'kill_storm_dragon_king_num', 'team_abbreviation'])

In [ ]:
data['data']['battle_player_list'][0]

{'team_id': '10017',
 'team_name': '广州TTG',
 'team_icon': 'https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823db5c8c44fae6abe.png',
 'hero_id': 503,
 'hero_name': '狂铁',
 'hero_icon': 'https://res.edata.qq.com/sgame/static/images/hero/503.jpg',
 'is_lose_mvp': 0,
 'camp': 1,
 'participation_rate': 75,
 'is_mvp': 0,
 'kill_num': 1,
 'assist_num': 5,
 'death_num': 1,
 'gold': 9210,
 'hurt_total': 235730,
 'be_hurt_total': 121009,
 'kda': 6,
 'hurt_total_rate': 0.1458,
 'be_hurt_total_rate': 0.3631,
 'mvp_score': 8.3,
 'actual_player_name': '广州TTG.萝卜',
 'SummonerAbilityInfo': {'summoner_ability_id': 80115,
  'summoner_ability_name': '闪现',
  'summoner_ability_rank': 'LV.19解锁',
  'summoner_ability_desc': '120秒CD：向指定方向位移一段距离',
  'summoner_ability_icon': 'https://res.edata.qq.com/sgame/static/images/summoner_ability/80115.jpg'},
 'BriefHeroSkillList': [{'skill_name': '力场压制',
   'skill_icon': 'https://res.edata.qq.com/sgame/static/images/hero_skill/503/50330.png'},
  {'skill_name': '强袭风暴',
   'skill

💡 **你的观察**（双击编辑这个 cell 写）：

- 顶层 keys 是：___message, code, request_id, data___
- `data['data']` 下二级 keys 大概有 _6__ 个
- 你**当下**就能说出含义的字段：__都可以____
- 你**猜不出**含义的字段（标记一下，后面查）：__无____

      "kill_big_dragon_num": 1,
      "push_tower_num": 5,
      "kill_dark_tyrant_num": 1,
      "kill_tyrant_num": 1,
      "kda": 7.5,
      "kill_prophet_dragon_num": 1,
      "kill_shadow_dragon_num": 1,
      "kill_storm_dragon_king_num": 0,
-
- 这里面两个字段出现错误，我刚刚查看比赛第一局的回放发现，腾讯在处理字段上出现问题，kill_shadow_dragon_num是暗影主宰的击杀数，kill_big_dragon_num是大龙的击杀数。目前来看我觉得可能是版本更新，原先大龙指的是先知主宰，但是后面引入暗影主宰，而一旦出现暗影主宰被击杀，那么大龙字段就会+1，而先知主宰也会+1。
- 我将解析第二个来查询是不是这个问题。

---
## 步骤 4 · 剥洋葱第二层：战队级数据（camp1 / camp2）

### 思路

**战队级数据是预测胜负的核心来源**——这一步要把每个字段的含义摸透。

技巧：把 camp1 和 camp2 **并排放进 DataFrame**。两队数据并排，差异一目了然，比一个个 print 高效 10 倍。

```python
pd.DataFrame([data['data']['camp1'], data['data']['camp2']])
```

In [ ]:
# TODO 4.1：先看看 camp1 有哪些字段
# 提示：data['data']['camp1'].keys()
data['data']['camp1'].keys()


dict_keys(['team_id', 'team_name', 'team_icon', 'is_win', 'kill_num', 'assist_num', 'death_num', 'gold', 'kill_big_dragon_num', 'push_tower_num', 'kill_dark_tyrant_num', 'kill_tyrant_num', 'kda', 'kill_prophet_dragon_num', 'kill_shadow_dragon_num', 'kill_storm_dragon_king_num', 'team_abbreviation'])

In [ ]:
# TODO 4.2：把 camp1 和 camp2 拼成 DataFrame，并排显示
# 提示：pd.DataFrame([...])，列表里两个 dict
# 进阶：用 .T 转置一下，让字段做行、阵营做列，更易读
df = pd.DataFrame([data['data']['camp1'],data['data']['camp2']]).T
df

,0,1
team_id,10017,10005
team_name,广州TTG,KSG
team_icon,https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823...,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...
is_win,True,False
kill_num,8,4
assist_num,22,9
death_num,4,8
gold,47042,39591
kill_big_dragon_num,1,0
push_tower_num,5,2


In [ ]:
data['data']['win_camp']

1

💡 **你的观察**：

请回答下面这几个问题（这些直接决定后面建模的特征选型）：

1. 战队级一共多少个字段？ 17
2. 哪些字段**对预测胜负有用**？（举 3 个）gold、kill_num、assist_num、push_tower_num、kill_dark_tyrant_num、kill_tyrant_num、kill_prophet_dragon_num、kill_shadow_dragon_num
3. 哪些字段**没分析价值**？（比如 logo URL）team_id、team_name、team_icon、team_abbreviation
4. 哪些字段是「赛后才有」的（比如人头数）？kda哪些是「赛前可知」的（比如战队 ID）？team_id、team_name、team_icon、team_abbreviation
5. 你看到 `win_camp` 的值是 1 还是 2？这决定了哪边赢——记住这个映射！win_camp = 1 蓝色方赢

---
## 步骤 5 · 剥洋葱第三层：选手级数据（battle_player_list）

### 思路

选手数据是 **list of dict** 结构——10 个选手，每人一个 dict。

处理这种结构有个 pandas 神器：**`pd.json_normalize`**。它能自动把嵌套字典展平成 DataFrame：

```python
pd.json_normalize(data['data']['battle_player_list'])
```

你之前 D 盘项目用过这个，但只用了表层。这次你会发现里面可能还有嵌套（比如英雄信息可能是个子 dict），`json_normalize` 也能处理——它会用 `.` 把嵌套键拼起来，比如 `hero.name`。

In [ ]:
# TODO 5.1：看下 battle_player_list 长什么样
# - 类型是什么？
# - 长度是多少？（KPL 是 5v5，应该是 10）
# - 第一个元素的所有 key？
battle_player_list_df = pd.json_normalize(data['data']['battle_player_list'])
battle_player_list_df

,team_id,team_name,team_icon,hero_id,hero_name,hero_icon,is_lose_mvp,camp,participation_rate,is_mvp,...,position_desc,hurt_to_hero_total,be_hurt_by_hero_total,hurt_to_hero_total_rate,be_hurt_by_hero_total_rate,SummonerAbilityInfo.summoner_ability_id,SummonerAbilityInfo.summoner_ability_name,SummonerAbilityInfo.summoner_ability_rank,SummonerAbilityInfo.summoner_ability_desc,SummonerAbilityInfo.summoner_ability_icon
0,10017,广州TTG,https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823...,503,狂铁,https://res.edata.qq.com/sgame/static/images/h...,0,1,75,0,...,对抗路,46492,91145,0.1333,0.3395,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
1,10017,广州TTG,https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823...,577,少司缘,https://res.edata.qq.com/sgame/static/images/h...,0,1,75,0,...,游走,37378,53607,0.1072,0.1997,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
2,10005,KSG,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...,126,夏侯惇,https://res.edata.qq.com/sgame/static/images/h...,0,2,25,0,...,打野,34084,138502,0.1269,0.3970,80104,惩击,LV.1解锁,30秒CD：对身边的野怪和小兵造成真1500点的实伤害并眩晕1秒,https://res.edata.qq.com/sgame/static/images/s...
3,10005,KSG,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...,177,苍,https://res.edata.qq.com/sgame/static/images/h...,1,2,100,0,...,发育路,103978,29364,0.3873,0.0842,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
4,10005,KSG,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...,136,武则天,https://res.edata.qq.com/sgame/static/images/h...,0,2,100,0,...,中路,79572,32187,0.2964,0.0923,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
5,10005,KSG,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...,518,马超,https://res.edata.qq.com/sgame/static/images/h...,0,2,25,0,...,对抗路,34021,43812,0.1267,0.1256,80109,疾跑,LV.7解锁,75秒CD：增加30%移动速度持续10秒，开启时移除自身的减速效果，且疾跑期间减少50%受到...,https://res.edata.qq.com/sgame/static/images/s...
6,10017,广州TTG,https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823...,519,敖隐,https://res.edata.qq.com/sgame/static/images/h...,0,1,75,0,...,发育路,88116,29198,0.2526,0.1087,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
7,10017,广州TTG,https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823...,178,杨戬,https://res.edata.qq.com/sgame/static/images/h...,0,1,75,1,...,打野,96470,70824,0.2765,0.2638,80104,惩击,LV.1解锁,30秒CD：对身边的野怪和小兵造成真1500点的实伤害并眩晕1秒,https://res.edata.qq.com/sgame/static/images/s...
8,10005,KSG,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...,194,苏烈,https://res.edata.qq.com/sgame/static/images/h...,0,2,75,0,...,游走,16842,104969,0.0627,0.3009,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
9,10017,广州TTG,https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823...,312,沈梦溪,https://res.edata.qq.com/sgame/static/images/h...,0,1,75,0,...,中路,80378,23723,0.2304,0.0884,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...


In [ ]:
battle_player_list_df.dtypes

team_id                                          str
team_name                                        str
team_icon                                        str
hero_id                                        int64
hero_name                                        str
hero_icon                                        str
is_lose_mvp                                    int64
camp                                           int64
participation_rate                             int64
is_mvp                                         int64
kill_num                                       int64
assist_num                                     int64
death_num                                      int64
gold                                           int64
hurt_total                                     int64
be_hurt_total                                  int64
kda                                          float64
hurt_total_rate                              float64
be_hurt_total_rate                           f

In [ ]:
len(battle_player_list_df)

10

In [ ]:
data['data']['battle_player_list'][0].keys()

dict_keys(['team_id', 'team_name', 'team_icon', 'hero_id', 'hero_name', 'hero_icon', 'is_lose_mvp', 'camp', 'participation_rate', 'is_mvp', 'kill_num', 'assist_num', 'death_num', 'gold', 'hurt_total', 'be_hurt_total', 'kda', 'hurt_total_rate', 'be_hurt_total_rate', 'mvp_score', 'actual_player_name', 'SummonerAbilityInfo', 'BriefHeroSkillList', 'BriefEquipList', 'player_name', 'player_icon', 'symbol_ids', 'position', 'position_desc', 'hurt_to_hero_total', 'be_hurt_by_hero_total', 'hurt_to_hero_total_rate', 'be_hurt_by_hero_total_rate'])

In [ ]:
# TODO 5.2：用 pd.json_normalize 一行展平
# 看 shape、看 columns、看 head()
battle_player_list_df.shape

(10, 37)

In [ ]:
battle_player_list_df.columns

Index(['team_id', 'team_name', 'team_icon', 'hero_id', 'hero_name',
       'hero_icon', 'is_lose_mvp', 'camp', 'participation_rate', 'is_mvp',
       'kill_num', 'assist_num', 'death_num', 'gold', 'hurt_total',
       'be_hurt_total', 'kda', 'hurt_total_rate', 'be_hurt_total_rate',
       'mvp_score', 'actual_player_name', 'BriefHeroSkillList',
       'BriefEquipList', 'player_name', 'player_icon', 'symbol_ids',
       'position', 'position_desc', 'hurt_to_hero_total',
       'be_hurt_by_hero_total', 'hurt_to_hero_total_rate',
       'be_hurt_by_hero_total_rate', 'SummonerAbilityInfo.summoner_ability_id',
       'SummonerAbilityInfo.summoner_ability_name',
       'SummonerAbilityInfo.summoner_ability_rank',
       'SummonerAbilityInfo.summoner_ability_desc',
       'SummonerAbilityInfo.summoner_ability_icon'],
      dtype='str')

In [ ]:
battle_player_list_df.head()

,team_id,team_name,team_icon,hero_id,hero_name,hero_icon,is_lose_mvp,camp,participation_rate,is_mvp,...,position_desc,hurt_to_hero_total,be_hurt_by_hero_total,hurt_to_hero_total_rate,be_hurt_by_hero_total_rate,SummonerAbilityInfo.summoner_ability_id,SummonerAbilityInfo.summoner_ability_name,SummonerAbilityInfo.summoner_ability_rank,SummonerAbilityInfo.summoner_ability_desc,SummonerAbilityInfo.summoner_ability_icon
0,10017,广州TTG,https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823...,503,狂铁,https://res.edata.qq.com/sgame/static/images/h...,0,1,75,0,...,对抗路,46492,91145,0.1333,0.3395,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
1,10017,广州TTG,https://smhtv-pic.tga.qq.com/0a05fdfba20fa3823...,577,少司缘,https://res.edata.qq.com/sgame/static/images/h...,0,1,75,0,...,游走,37378,53607,0.1072,0.1997,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
2,10005,KSG,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...,126,夏侯惇,https://res.edata.qq.com/sgame/static/images/h...,0,2,25,0,...,打野,34084,138502,0.1269,0.3970,80104,惩击,LV.1解锁,30秒CD：对身边的野怪和小兵造成真1500点的实伤害并眩晕1秒,https://res.edata.qq.com/sgame/static/images/s...
3,10005,KSG,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...,177,苍,https://res.edata.qq.com/sgame/static/images/h...,1,2,100,0,...,发育路,103978,29364,0.3873,0.0842,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...
4,10005,KSG,https://smobatv-pic.tga.qq.com/e7f3ca48aa29a89...,136,武则天,https://res.edata.qq.com/sgame/static/images/h...,0,2,100,0,...,中路,79572,32187,0.2964,0.0923,80115,闪现,LV.19解锁,120秒CD：向指定方向位移一段距离,https://res.edata.qq.com/sgame/static/images/s...


In [ ]:
# TODO 5.3：列名太多看不过来？挑感兴趣的列出来看
# 思路：先 print 全部列名，再列表索引几个核心列
# 比如：玩家名 / 英雄 / 阵营 / kill / death / assist / money_total / damage
print(battle_player_list_df.columns)
battle_player_list_df['BriefEquipList']

Index(['team_id', 'team_name', 'team_icon', 'hero_id', 'hero_name',
       'hero_icon', 'is_lose_mvp', 'camp', 'participation_rate', 'is_mvp',
       'kill_num', 'assist_num', 'death_num', 'gold', 'hurt_total',
       'be_hurt_total', 'kda', 'hurt_total_rate', 'be_hurt_total_rate',
       'mvp_score', 'actual_player_name', 'BriefHeroSkillList',
       'BriefEquipList', 'player_name', 'player_icon', 'symbol_ids',
       'position', 'position_desc', 'hurt_to_hero_total',
       'be_hurt_by_hero_total', 'hurt_to_hero_total_rate',
       'be_hurt_by_hero_total_rate', 'SummonerAbilityInfo.summoner_ability_id',
       'SummonerAbilityInfo.summoner_ability_name',
       'SummonerAbilityInfo.summoner_ability_rank',
       'SummonerAbilityInfo.summoner_ability_desc',
       'SummonerAbilityInfo.summoner_ability_icon'],
      dtype='str')


0    [{'equip_id': 11311, 'equip_name': '纯净苍穹', 'eq...
1    [{'equip_id': 1723, 'equip_name': '极影·奔狼', 'eq...
2    [{'equip_id': 1532, 'equip_name': '巨人之握', 'equ...
3    [{'equip_id': 1133, 'equip_name': '无尽战刃', 'equ...
4    [{'equip_id': 1423, 'equip_name': '冷静之靴', 'equ...
5    [{'equip_id': 1422, 'equip_name': '抵抗之靴', 'equ...
6    [{'equip_id': 1133, 'equip_name': '无尽战刃', 'equ...
7    [{'equip_id': 1533, 'equip_name': '贪婪之噬', 'equ...
8    [{'equip_id': 1724, 'equip_name': '近卫·救赎', 'eq...
9    [{'equip_id': 12211, 'equip_name': '梦魇之牙', 'eq...
Name: BriefEquipList, dtype: object

💡 **你的观察**：

1. 怎么区分**位置**（对抗 / 打野 / 中路 / 发育 / 游走）？字段名是？字段名是position_desc，值：对抗路 / 打野 / 中路 / 发育路 / 游走
2. 怎么区分**蓝方 / 红方**？（提示：可能有个 camp 字段）camp = 1 是蓝色方；camp=2 是红色方
3. 英雄 ID 和英雄名字段叫什么？hero_name和hero_id
4. KDA 三个值（kill / death / assist）字段名是？kill_num、death_num、assist_num
5. 有没有「装备」「召唤师技能」「铭文」相关字段？这些将来能做出彩的分析BriefEquipList装备、SummonerAbilityInfo召唤师技能、BriefHeroSkillList英雄技能、没找到铭文

---
## 步骤 6 · 剥洋葱第四层：BP 数据（bp_list）

### 思路

**BP（Ban / Pick，禁用 / 选用）是赛前可知的最重要信息之一**——比赛还没开打，BP 就已经决定了。

为什么 BP 重要？
- **战术信号**：选了什么阵容，体现了战队对版本的理解
- **强度差**：版本强势英雄被先选，胜率天然就高一些
- **赛前可用**：建模时它属于「合法特征」，不会数据泄漏

结构上 bp_list 也是 list of dict，但每个元素只代表「一次操作」（一次 ban 或一次 pick），需要看：
- 哪个字段区分 ban / pick？
- 哪个字段表示是哪一队操作？
- 顺序是怎么排的？

In [ ]:
# TODO 6.1：看 bp_list 的基本信息
# - 长度多少？（一般 BO 单局约 6 个 ban + 10 个 pick = 16 条左右）
# - 第一个元素的 keys？
len(data['data']['bp_list'])

20

In [ ]:
data['data']['bp_list'][0].keys()

dict_keys(['camp', 'is_ban_or_pick', 'hero_id', 'hero_name', 'hero_icon', 'position'])

In [ ]:
# TODO 6.2：展平成 DataFrame
# 重点关注：
#   - action / type 字段：怎么区分 ban / pick？
#   - camp 字段：哪一队？
#   - sequence / order 字段：先后顺序
#   - hero 字段：英雄是用 ID 还是中文名？
pd.json_normalize(data['data']['bp_list'])


,camp,is_ban_or_pick,hero_id,hero_name,hero_icon,position
0,1,0,521,海月,https://res.edata.qq.com/sgame/static/images/h...,0
1,2,0,525,鲁班大师,https://res.edata.qq.com/sgame/static/images/h...,0
2,1,0,517,大司命,https://res.edata.qq.com/sgame/static/images/h...,0
3,2,0,581,元流之子(坦克),https://res.edata.qq.com/sgame/static/images/h...,0
4,1,1,519,敖隐,https://res.edata.qq.com/sgame/static/images/h...,0
5,2,1,194,苏烈,https://res.edata.qq.com/sgame/static/images/h...,1
6,2,1,177,苍,https://res.edata.qq.com/sgame/static/images/h...,0
7,1,1,503,狂铁,https://res.edata.qq.com/sgame/static/images/h...,1
8,1,1,577,少司缘,https://res.edata.qq.com/sgame/static/images/h...,2
9,2,1,518,马超,https://res.edata.qq.com/sgame/static/images/h...,2


💡 **你的观察**：

1. ban 和 pick 怎么区分？字段值是什么？
2. 阵营怎么标？camp = 1是蓝色方 ，camp=2是红色方
3. 顺序对不对得上 KPL 的 BP 流程（4 ban / 3 pick / 2 ban / 2 pick / 3 pick）？position  0 ban /1pick/2ban/3pick/4pick BP就按照索引来的，蓝色ban一个，红色ban一个。双方先ban两个。然后蓝色方先选一个，红两个蓝两个，红一个，然后ban，红色先ban，轮流一个，然后选英雄红一个，蓝两个，红一个
4. 英雄能直接看到中文名，还是需要查字典转换？能看到中文，不需要字典

---
## 步骤 7 · 验证 battle_id 编码规则 ⭐⭐⭐

### 思路

**这是整个 notebook 最重要的实验**。

你之前的发现：

```
battle_id = 战队A_id_局数_战队B_id
例：1038107152_18_1742644777
```

但这是**猜测**，没验证过。我们要做 3 个实验：

| 实验 | 操作 | 想验证什么 |
|------|------|----------|
| A 战队互换 | A_id 和 B_id 调换位置再调接口 | 编码顺序敏感吗？两个 ID 哪个是「主队」？ |
| B 局数 +1 | 18 改成 19、20、21 | 是不是 BO 系列下一局？数据是否真的不同？ |
| C 故意错误 | 局数改成 999 | 接口怎么报错？code 是几？ |

**为什么这一步重要？**

如果编码规则验证成功了，你就能**用 Python 批量构造 URL**——给定一场 BO5/BO7，算出所有局的 battle_id，不用依赖赛程页爬虫。这是工程上的巨大便利。

如果失败，说明必须老老实实从赛程接口拉 battle_id 列表（步骤 8）。

In [59]:
# TODO 7.0：先写一个小工具函数，避免重复代码
# def quick_get(battle_id):
#     """传入 battle_id，返回 (code, win_camp, kill_a, kill_b)"""
#     ... 调接口 ...
#     return ...
# 这样下面三个实验都用这个函数，对比清晰

def quick_get(battle_id):
	url = f"{BASE_URL}?battle_id={battle_id}"
	resp =  requests.get(url,headers=HEADERS,timeout=15)
	data = resp.json()
	return data['code']

In [83]:
# TODO 7.1【实验 A · 战队互换】
# 原 ID：1038107152_18_1742644777  （或换成你的 TEST_BATTLE_ID）
# 新 ID：1742644777_18_1038107152
# 调接口看：
#   - 能不能成功（code = 0）？
#   - 是同一场比赛吗（对比 win_camp、人头数）？
#   - camp1 和 camp2 调换了吗？

quick_get('1742644777_18_1038107152')

404

In [63]:
# TODO 7.2【实验 B · 局数变化】
# 把局数 18 → 19 → 20 → 21 → 22
# 看哪些能拉到、哪些是 "局数不存在" 的报错
# 这告诉你这场 BO 一共打了几局、第几局是开始

for game_num in [10,11,12,13,14,15,16,17,18,19,20]:
    battle_id = f"736117264_{game_num}_1777089898"
    print(quick_get(battle_id))
    pass


200
404
404
404
404
404
404
404
404
404
404


In [64]:
# TODO 7.3【实验 C · 故意错误】
# 把局数改成 999（绝对不存在）
# 看接口的 code、message 是什么
# 这决定了将来 fetch_battle 怎么判断 "这条 ID 是无效的"
quick_get('1742644777_99_1038107152')


404

💡 **你的实验结论**（必填，这是简历项目的核心思考）：

**实验 A · 战队互换**：
- 能不能拉到数据？
- 是同一场吗？camp1/camp2 顺序是否互换？
- 结论：编码顺序的语义是 ___固定的___

**实验 B · 局数变化**：
- 这场 BO 一共有几局？
- 局数从几开始（10？18？）？
- 结论：局数 ID 的取值范围 ___失败，因为battle_id每一句会变，中间确实是+1顺序，但是后面的数字会变动___

**实验 C · 故意错误**：
- 接口 code 返回 ___404___
- message 内容 __空____

**业务意义（最重要）**：根据上面 3 个结论，回答下面问题：
- ✅ 我们能不能用规则**批量构造** battle_id？ 不能，目前不清楚，battle_id：前中后数字含义
- ❌ 还是必须从赛程接口拉 ID 列表？

---
## 步骤 8 · 找赛程接口

### 思路

无论步骤 7 的结论是什么，赛程接口都很重要——它能告诉我们：
- 整个赛季有哪些比赛
- 哪些已经打完、哪些还没开始
- 哪两支队对阵

找接口的方法有两种：

**方法一（推荐先试）：直接请求赛程 HTML**

如果是「服务端渲染」（SSR），HTML 里就直接有比赛数据，正则或 BeautifulSoup 一抠就行。

**方法二：F12 抓异步接口**

如果方法一拉到的 HTML 里没数据（说明数据是 JS 异步加载的），就要去浏览器抓包：

1. 打开 https://pvp.qq.com/matchdata/schedule.html?league_id=20260002
2. F12 → Network → 勾选 XHR/Fetch
3. 刷新页面，看哪个请求返回了赛程 JSON
4. 复制 URL 和 Headers 回来这里测试

In [ ]:
# TODO 8.1【方法一】请求赛程页 HTML
# url = 'https://pvp.qq.com/matchdata/schedule.html?league_id=20260002'
# 拉下来后做几个判断：
#   - len(text) 多大？小于 50KB 大概率是空壳，是 JS 渲染
#   - '战队' 这种词在 HTML 里搜得到吗？
#   - 'battle_id' 在 HTML 里搜得到吗？



搜不到

In [ ]:
# TODO 8.2【方法二，如果 8.1 失败再做】
# 在浏览器 F12 里找到那个返回 JSON 的接口 URL
# 把 URL 和它需要的 Headers 抄过来，在这里测试请求
# 看看 JSON 结构里有没有 battle_id



已完成

💡 **你的发现**：

1. 赛程页是 SSR 还是异步加载？异步
2. 如果是异步，接口 URL 是？
3. 接口返回的 JSON 里能不能直接拿到 battle_id？能
4. 一个赛季大概有多少场比赛？

- url:https://prod.comp.smoba.qq.com/leaguesite/leagues/open  是历史比赛主题，里面获取league_id
- url:
https://prod.comp.smoba.qq.com/leaguesite/matches/open?league_id=20260002 是这个比赛主题下面的赛程，match_id用于来确定某一场里面的BO3/5/7里面比赛id
- url:https://prod.comp.smoba.qq.com/leaguesite/match/battles/open?match_id=2026050302 是一场比赛里面的局数里面有battle_id
- 然后通过battle_id来爬取

---
## 步骤 9 · 沉淀字段字典（最容易偷懒的一步）

### 思路

前面 1-8 步是「探索」，这一步是**「沉淀」**——把脑子里的发现写成结构化文档。

**为什么不能省？**

你 D 盘旧项目最大的痛点：半年后回来看代码，不记得字段是什么意思，只能重新探索一遍。

字段字典就是给「未来的你」（和未来面试官）看的说明书。

### 任务

打开 `docs/数据字典.md`，至少补全 4 张表：

| 表 | 来源 | 字段数预计 |
|----|------|-----------|
| 比赛级 | `data['data']` 直接的字段 | 10-15 个 |
| 战队级 | `camp1` / `camp2` 内字段 | 10-20 个 |
| 选手级 | `battle_player_list` 内字段 | 30-50 个 |
| BP 级 | `bp_list` 内字段 | 4-6 个 |

每张表至少 4 列：**字段名 / 类型 / 含义 / 示例值**

进阶：再加一列「赛前可知 / 赛后才有」——这是后面建模时**判断能不能用**的关键标记。

---
## ✅ 完成自检

做完一项打一个 √：

- [ ] `data/raw/sample.json` 已保存（步骤 2）
- [ ] 顶层结构已摸清（步骤 3 的 💡 写完了）
- [ ] 战队级字段已摸清（步骤 4 的 💡 写完了）
- [ ] 选手级字段已摸清（步骤 5 的 💡 写完了）
- [ ] BP 级字段已摸清（步骤 6 的 💡 写完了）
- [ ] battle_id 编码规则的 3 个实验都做了，结论写了（步骤 7）
- [ ] 找到赛程接口（或确认了 HTML 能解析）（步骤 8）
- [ ] `docs/数据字典.md` 至少 4 张表填完了（步骤 9）

## 🎯 完成后跟我汇报

用这个格式（和项目 01 一样）：

```
1. 我做了什么：______
2. 我的思考：______（至少 2 条业务理解）
3. 我想确认：______（卡住或不确定的地方）
```